In [4]:
# Load the SQL toolset extension
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [5]:
#2. Connect to (and create) the database file
%sql sqlite:///isabel_week5_workbook.db

Connecting to 'sqlite:///isabel_week5_workbook.db'

### Create the table AUTHORS

In [6]:
%%sql
CREATE TABLE Authors (
    author_id  INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name  TEXT,
    birth_year INTEGER,
    country    TEXT
);

Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Authors already exists
[SQL: CREATE TABLE Authors (
    author_id  INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name  TEXT,
    birth_year INTEGER,
    country    TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Create the table BOOKS

In [7]:
%%sql
CREATE TABLE Books (
    book_id          INTEGER PRIMARY KEY,
    title            TEXT,
    author_id        INTEGER REFERENCES Authors (author_id),
    genre_id         INTEGER REFERENCES Genres (genre_id),
    isbn             TEXT,
    publication_year TEXT,
    copies_owned     TEXT
);


Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Books already exists
[SQL: CREATE TABLE Books (
    book_id          INTEGER PRIMARY KEY,
    title            TEXT,
    author_id        INTEGER REFERENCES Authors (author_id),
    genre_id         INTEGER REFERENCES Genres (genre_id),
    isbn             TEXT,
    publication_year TEXT,
    copies_owned     TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Create the table BRANCHES

In [8]:
%%sql
CREATE TABLE Books (
    book_id          INTEGER PRIMARY KEY,
    title            TEXT,
    author_id        INTEGER REFERENCES Authors (author_id),
    genre_id         INTEGER REFERENCES Genres (genre_id),
    isbn             TEXT,
    publication_year TEXT,
    copies_owned     TEXT
);


Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Books already exists
[SQL: CREATE TABLE Books (
    book_id          INTEGER PRIMARY KEY,
    title            TEXT,
    author_id        INTEGER REFERENCES Authors (author_id),
    genre_id         INTEGER REFERENCES Genres (genre_id),
    isbn             TEXT,
    publication_year TEXT,
    copies_owned     TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Create the table GENRES

In [9]:
%%sql
CREATE TABLE Genres (
    genre_id    INTEGER PRIMARY KEY,
    genre_name  TEXT,
    description TEXT
);


Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Genres already exists
[SQL: CREATE TABLE Genres (
    genre_id    INTEGER PRIMARY KEY,
    genre_name  TEXT,
    description TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Insert values into the LOANS table

In [10]:
%%sql
CREATE TABLE Loans (
    loan_id       INTEGER PRIMARY KEY,
    book_id       INTEGER REFERENCES Books (book_id),
    patron_id     INTEGER REFERENCES Patrons (patron_id),
    branch_id     INTEGER REFERENCES Branches (branch_id),
    checkout_date TEXT,
    due_date      TEXT,
    return_date   TEXT
);

Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Loans already exists
[SQL: CREATE TABLE Loans (
    loan_id       INTEGER PRIMARY KEY,
    book_id       INTEGER REFERENCES Books (book_id),
    patron_id     INTEGER REFERENCES Patrons (patron_id),
    branch_id     INTEGER REFERENCES Branches (branch_id),
    checkout_date TEXT,
    due_date      TEXT,
    return_date   TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Insert values into the PATRONS table

In [11]:
%%sql
CREATE TABLE Patrons (
    patron_id         INTEGER PRIMARY KEY,
    first_name        TEXT,
    last_name         TEXT,
    email             TEXT,
    address           TEXT,
    city              TEXT,
    registration_date TEXT,
    branch_id         INTEGER REFERENCES Branches (branch_id) 
);


Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: (sqlite3.OperationalError) table Patrons already exists
[SQL: CREATE TABLE Patrons (
    patron_id         INTEGER PRIMARY KEY,
    first_name        TEXT,
    last_name         TEXT,
    email             TEXT,
    address           TEXT,
    city              TEXT,
    registration_date TEXT,
    branch_id         INTEGER REFERENCES Branches (branch_id)
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Part 1: Basic SQL Operations and JOIN Queries

### 1. Books published after 2000

In [12]:
%%sql
SELECT title, publication_year
FROM books
WHERE publication_year > 2000
ORDER BY publication_year DESC;

Running query in 'sqlite:///isabel_week5_workbook.db'

title,publication_year


### 2. Books with more than 5 copies in fiction genre

In [13]:
%%sql
SELECT *
FROM books
WHERE copies_owned > 5
  AND genre_id = 1;

Running query in 'sqlite:///isabel_week5_workbook.db'

book_id,title,author_id,genre_id,isbn,publication_year,copies_owned


### 3. Books with “History” in the title

In [14]:
%%sql
SELECT *
FROM books
WHERE title LIKE '%History%';

Running query in 'sqlite:///isabel_week5_workbook.db'

book_id,title,author_id,genre_id,isbn,publication_year,copies_owned


### 4. Loans made in January 2023 with patron details

In [15]:
%%sql
SELECT 
    loans.loan_id,
    loans.checkout_date,
    loans.due_date,
    patrons.first_name,
    patrons.last_name,
    patrons.email
FROM loans
JOIN patrons
    ON loans.patron_id = patrons.patron_id
WHERE loans.checkout_date BETWEEN '2023-01-01' AND '2023-01-31';

Running query in 'sqlite:///isabel_week5_workbook.db'

loan_id,checkout_date,due_date,first_name,last_name,email


### 5. Book details for each loan

In [16]:
%%sql
SELECT 
    books.title,
    authors.first_name || ' ' || authors.last_name AS author_name,
    genres.genre_name,
    loans.checkout_date,
    loans.due_date
FROM loans
JOIN books
    ON loans.book_id = books.book_id
JOIN authors
    ON books.author_id = authors.author_id
JOIN genres
    ON books.genre_id = genres.genre_id;

Running query in 'sqlite:///isabel_week5_workbook.db'

title,author_name,genre_name,checkout_date,due_date


### 6. Pairs of patrons who live in the same city

In [17]:
%%sql
SELECT 
    p1.first_name || ' ' || p1.last_name AS patron_1,
    p2.first_name || ' ' || p2.last_name AS patron_2,
    p1.city
FROM patrons p1
JOIN patrons p2
    ON p1.city = p2.city
   AND p1.patron_id < p2.patron_id;

Running query in 'sqlite:///isabel_week5_workbook.db'

patron_1,patron_2,city


### 7. Fiction books borrowed, with patron and branch

In [18]:
%%sql
SELECT 
    books.title,
    patrons.first_name || ' ' || patrons.last_name AS patron_name,
    branches.branch_name
FROM loans
JOIN books
    ON loans.book_id = books.book_id
JOIN patrons
    ON loans.patron_id = patrons.patron_id
JOIN branches
    ON loans.branch_id = branches.branch_id
WHERE books.genre_id = 1;

Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(sqlite3.OperationalError) no such table: branches
[SQL: SELECT
    books.title,
    patrons.first_name || ' ' || patrons.last_name AS patron_name,
    branches.branch_name
FROM loans
JOIN books
    ON loans.book_id = books.book_id
JOIN patrons
    ON loans.patron_id = patrons.patron_id
JOIN branches
    ON loans.branch_id = branches.branch_id
WHERE books.genre_id = 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)



### Part 2: Aggregation and GROUP BY

### 8. Count books in each genre

In [19]:
%%sql
SELECT 
    genres.genre_name,
    COUNT(books.book_id) AS number_of_books
FROM genres
JOIN books
    ON genres.genre_id = books.genre_id
GROUP BY genres.genre_name;

Running query in 'sqlite:///isabel_week5_workbook.db'

genre_name,number_of_books


### 9. Average, minimum, and maximum loan duration by branch

In [20]:
%%sql
SELECT 
    branches.branch_name,
    AVG(julianday(loans.return_date) - julianday(loans.checkout_date)) AS average_loan_duration,
    MIN(julianday(loans.return_date) - julianday(loans.checkout_date)) AS minimum_loan_duration,
    MAX(julianday(loans.return_date) - julianday(loans.checkout_date)) AS maximum_loan_duration
FROM loans
JOIN branches
    ON loans.branch_id = branches.branch_id
WHERE loans.return_date IS NOT NULL
GROUP BY branches.branch_name;

Running query in 'sqlite:///isabel_week5_workbook.db'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(sqlite3.OperationalError) no such table: branches
[SQL: SELECT
    branches.branch_name,
    AVG(julianday(loans.return_date) - julianday(loans.checkout_date)) AS average_loan_duration,
    MIN(julianday(loans.return_date) - julianday(loans.checkout_date)) AS minimum_loan_duration,
    MAX(julianday(loans.return_date) - julianday(loans.checkout_date)) AS maximum_loan_duration
FROM loans
JOIN branches
    ON loans.branch_id = branches.branch_id
WHERE loans.return_date IS NOT NULL
GROUP BY branches.branch_name;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)



### 10. Patrons with overdue books

In [21]:
%%sql
SELECT 
    patrons.first_name,
    patrons.last_name,
    COUNT(loans.loan_id) AS overdue_books
FROM loans
JOIN patrons
    ON loans.patron_id = patrons.patron_id
WHERE loans.due_date < date('now')
  AND loans.return_date IS NULL
GROUP BY patrons.patron_id, patrons.first_name, patrons.last_name;

Running query in 'sqlite:///isabel_week5_workbook.db'

first_name,last_name,overdue_books


In [22]:
If your blank return dates imported as empty spaces instead of NULL, use this version:

SyntaxError: invalid syntax (3291980903.py, line 1)

In [23]:
%%sql
SELECT 
    patrons.first_name,
    patrons.last_name,
    COUNT(loans.loan_id) AS overdue_books
FROM loans
JOIN patrons
    ON loans.patron_id = patrons.patron_id
WHERE loans.due_date < date('now')
  AND loans.return_date = ''
GROUP BY patrons.patron_id, patrons.first_name, patrons.last_name;

Running query in 'sqlite:///isabel_week5_workbook.db'

first_name,last_name,overdue_books


### 11. Monthly borrowing trends

In [24]:
%%sql
SELECT 
    strftime('%Y', checkout_date) AS year,
    strftime('%m', checkout_date) AS month,
    COUNT(loan_id) AS number_of_loans,
    COUNT(DISTINCT patron_id) AS unique_patrons
FROM loans
GROUP BY year, month
ORDER BY year, month;

Running query in 'sqlite:///isabel_week5_workbook.db'

year,month,number_of_loans,unique_patrons


In [ ]:
# Close the connection
